## Example 1: blinker functionality test 1

In [ ]:
from blinker import signal

# Example 1: Basic signal creation and connection with Blinker
print("Example 1: Basic signal creation and connection with Blinker")

try:
    # Create a named signal
    user_signal = signal('user-action')
    
    print("✓ Blinker signal library imported")
    print("Creating and connecting signals:")
    
    # Define signal handlers
    def on_user_login(sender, username=None, **kwargs):
        print(f"  Handler 1: User '{username}' logged in from {sender}")
    
    def on_user_activity(sender, username=None, **kwargs):
        print(f"  Handler 2: Logging activity for user '{username}'")
    
    # Connect handlers to signal
    user_signal.connect(on_user_login)
    user_signal.connect(on_user_activity)
    
    print(f"✓ Signal created: {user_signal}")
    print(f"✓ Connected {len(user_signal.receivers)} handlers")
    
    # Send signal with data
    print("\nSending signals:")
    user_signal.send('web-app', username='alice')
    user_signal.send('mobile-app', username='bob')
    
    # Show signal information
    print(f"\nSignal information:")
    print(f"Signal name: {user_signal.name}")
    print(f"Number of receivers: {len(user_signal.receivers)}")
    print(f"Has receivers: {bool(user_signal.receivers)}")
    
    print("✓ Basic signal creation and connection completed")
    print("blinker example 1 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blinker example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: blinker functionality test 2

In [ ]:
from blinker import Namespace

# Example 2: Signal namespaces and advanced connection patterns
print("Example 2: Signal namespaces and advanced connection patterns")

try:
    # Create a signal namespace
    app_signals = Namespace()
    
    # Define multiple signals in the namespace
    database_signal = app_signals.signal('database-event')
    cache_signal = app_signals.signal('cache-event')
    auth_signal = app_signals.signal('auth-event')
    
    print("✓ Signal namespace created with multiple signals")
    print(f"Namespace signals: {len(app_signals)} defined")
    
    # Advanced connection patterns
    event_log = []
    
    # Connect with sender filtering
    def database_handler(sender, operation=None, table=None, **kwargs):
        event_log.append(f"DB: {operation} on {table} from {sender}")
        print(f"  Database event: {operation} on {table}")
    
    def cache_handler(sender, action=None, key=None, **kwargs):
        event_log.append(f"Cache: {action} key '{key}' from {sender}")
        print(f"  Cache event: {action} key '{key}'")
    
    def auth_handler(sender, event_type=None, user=None, **kwargs):
        event_log.append(f"Auth: {event_type} for {user} from {sender}")
        print(f"  Auth event: {event_type} for user {user}")
    
    # Connect handlers
    database_signal.connect(database_handler)
    cache_signal.connect(cache_handler)
    auth_signal.connect(auth_handler)
    
    # Send various signals
    print("\nSending namespace signals:")
    database_signal.send('orm-layer', operation='SELECT', table='users')
    database_signal.send('migration-tool', operation='CREATE', table='sessions')
    
    cache_signal.send('redis-client', action='SET', key='user:123')
    cache_signal.send('memcached-client', action='DELETE', key='session:abc')
    
    auth_signal.send('login-service', event_type='login', user='alice')
    auth_signal.send('oauth-provider', event_type='token_refresh', user='bob')
    
    # Conditional connections with weak references
    print(f"\nAdvanced connection features:")
    
    # Connect with sender-specific filtering
    def specific_sender_handler(sender, **kwargs):
        print(f"  Handler only for 'special-sender': {kwargs}")
    
    # Connect only for specific sender
    database_signal.connect(specific_sender_handler, sender='special-sender', weak=False)
    
    # Test sender-specific connection
    database_signal.send('regular-sender', operation='UPDATE', table='posts')
    database_signal.send('special-sender', operation='DELETE', table='comments')
    
    # Show event log summary
    print(f"\nEvent log summary:")
    print(f"Total events captured: {len(event_log)}")
    for event in event_log[-3:]:  # Show last 3 events
        print(f"  Recent: {event}")
    
    # Signal disconnection
    print(f"\nSignal management:")
    print(f"Database signal receivers: {len(database_signal.receivers)}")
    
    # Disconnect a handler
    database_signal.disconnect(database_handler)
    print(f"After disconnect: {len(database_signal.receivers)} receivers")
    
    print("✓ Signal namespaces and advanced patterns completed")
    print("blinker example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blinker example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: blinker functionality test 3

In [ ]:
from blinker import signal

# Example 3: Signal decorators and context managers
print("Example 3: Signal decorators and context managers")

try:
    # Create signals for different events
    before_request = signal('before-request')
    after_request = signal('after-request')
    error_occurred = signal('error-occurred')
    
    print("✓ Created event signals for request lifecycle")
    
    # Decorator-based signal connections
    @before_request.connect
    def log_request_start(sender, request_id=None, method=None, path=None, **kwargs):
        print(f"  → Starting {method} {path} (ID: {request_id})")
    
    @after_request.connect  
    def log_request_end(sender, request_id=None, status_code=None, duration=None, **kwargs):
        print(f"  ← Finished request {request_id}: {status_code} ({duration}ms)")
    
    @error_occurred.connect
    def handle_error(sender, request_id=None, error_type=None, message=None, **kwargs):
        print(f"  ✗ Error in request {request_id}: {error_type} - {message}")
    
    print("✓ Connected handlers using decorators")
    
    # Simulate request processing with signals
    def process_request(request_id, method, path, should_error=False):
        import random
        
        # Send before request signal
        before_request.send('web-server', 
                          request_id=request_id,
                          method=method, 
                          path=path)
        
        # Simulate processing time
        duration = random.randint(50, 500)
        
        if should_error:
            # Send error signal
            error_occurred.send('request-processor',
                              request_id=request_id,
                              error_type='ValidationError',
                              message='Invalid input data')
            status_code = 400
        else:
            status_code = 200
        
        # Send after request signal
        after_request.send('web-server',
                         request_id=request_id, 
                         status_code=status_code,
                         duration=duration)
        
        return status_code
    
    print("\nProcessing sample requests:")
    
    # Process successful requests
    process_request('req-001', 'GET', '/api/users')
    process_request('req-002', 'POST', '/api/users', should_error=False)
    process_request('req-003', 'PUT', '/api/users/123', should_error=True)
    process_request('req-004', 'DELETE', '/api/users/456')
    
    # Multiple handlers for same signal
    print(f"\nMultiple handlers demonstration:")
    
    # Add additional handlers
    @before_request.connect
    def security_check(sender, **kwargs):
        print(f"    Security: Checking permissions for request")
    
    @before_request.connect  
    def rate_limit_check(sender, **kwargs):
        print(f"    Rate limit: Checking request limits")
    
    print(f"Before request now has {len(before_request.receivers)} handlers")
    
    # Process request to show multiple handlers
    print(f"\nRequest with multiple handlers:")
    process_request('req-005', 'GET', '/api/protected')
    
    # Signal context and results
    print(f"\nSignal results and context:")
    
    # Handler that returns results
    results = []
    
    @after_request.connect
    def collect_metrics(sender, **kwargs):
        result = f"Metrics collected for {kwargs.get('request_id')}"
        results.append(result)
        return result
    
    # Process request and collect results
    process_request('req-006', 'GET', '/api/metrics')
    
    print(f"Collected results: {len(results)} items")
    for result in results:
        print(f"  • {result}")
    
    # Signal introspection
    print(f"\nSignal introspection:")
    signals_info = [
        ('before_request', before_request),
        ('after_request', after_request), 
        ('error_occurred', error_occurred)
    ]
    
    for name, sig in signals_info:
        print(f"  {name}: {len(sig.receivers)} receivers, name='{sig.name}'")
    
    print("✓ Signal decorators and context managers completed")
    print("blinker example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blinker example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: blinker functionality test 4

In [ ]:
from blinker import Signal
import weakref

# Example 4: Weak references, memory management and signal lifecycle
print("Example 4: Weak references, memory management and signal lifecycle")

try:
    # Create custom signal class for demonstration
    lifecycle_signal = Signal()
    connection_signal = Signal()
    
    print("✓ Created signals for lifecycle management")
    
    # Class-based handlers to test weak references
    class EventLogger:
        def __init__(self, name):
            self.name = name
            self.events = []
        
        def handle_lifecycle(self, sender, event_type=None, **kwargs):
            message = f"{self.name}: {event_type} from {sender}"
            self.events.append(message)
            print(f"  Logger '{self.name}': {event_type}")
        
        def handle_connection(self, sender, action=None, **kwargs):
            message = f"{self.name}: Connection {action}"
            self.events.append(message)
            print(f"  Connection '{self.name}': {action}")
    
    # Create logger instances
    logger1 = EventLogger("Primary")
    logger2 = EventLogger("Secondary")
    logger3 = EventLogger("Temporary")
    
    print("✓ Created event logger instances")
    
    # Connect with different weak reference settings
    print("\nConnecting handlers with weak reference management:")
    
    # Strong reference (default weak=True, but we'll set weak=False for some)
    lifecycle_signal.connect(logger1.handle_lifecycle, weak=False)
    connection_signal.connect(logger1.handle_connection, weak=False)
    print("  • Primary logger: strong references")
    
    # Weak reference (default behavior)
    lifecycle_signal.connect(logger2.handle_lifecycle, weak=True)
    connection_signal.connect(logger2.handle_connection, weak=True)  
    print("  • Secondary logger: weak references")
    
    # Temporary connection (weak reference)
    lifecycle_signal.connect(logger3.handle_lifecycle, weak=True)
    print("  • Temporary logger: weak references")
    
    # Show initial connection count
    print(f"\nInitial connections:")
    print(f"  Lifecycle signal: {len(lifecycle_signal.receivers)} receivers")
    print(f"  Connection signal: {len(connection_signal.receivers)} receivers")
    
    # Send signals to test all connections
    print("\nTesting initial signal delivery:")
    lifecycle_signal.send('app-start', event_type='startup')
    connection_signal.send('database', action='connected')
    
    # Delete temporary logger to test weak references
    print(f"\nDeleting temporary logger...")
    del logger3
    
    # Force garbage collection to clean up weak references
    import gc
    gc.collect()
    
    print(f"After cleanup:")
    print(f"  Lifecycle signal: {len(lifecycle_signal.receivers)} receivers")
    print(f"  Connection signal: {len(connection_signal.receivers)} receivers")
    
    # Send signals again to see effect of cleanup
    print(f"\nTesting after cleanup:")
    lifecycle_signal.send('app-running', event_type='active')
    connection_signal.send('cache', action='initialized')
    
    # Manual disconnection
    print(f"\nManual disconnection:")
    lifecycle_signal.disconnect(logger2.handle_lifecycle)
    connection_signal.disconnect(logger2.handle_connection)
    
    print(f"After manual disconnect:")
    print(f"  Lifecycle signal: {len(lifecycle_signal.receivers)} receivers")
    print(f"  Connection signal: {len(connection_signal.receivers)} receivers")
    
    # Test with only strong references remaining
    print(f"\nTesting with remaining strong references:")
    lifecycle_signal.send('app-shutdown', event_type='cleanup')
    connection_signal.send('database', action='disconnected')
    
    # Show event logs
    print(f"\nEvent logs summary:")
    print(f"  Primary logger events: {len(logger1.events)}")
    for event in logger1.events[-3:]:  # Show last 3
        print(f"    Recent: {event}")
    
    print(f"  Secondary logger events: {len(logger2.events)}")
    for event in logger2.events[-2:]:  # Show last 2
        print(f"    Recent: {event}")
    
    # Signal cleanup and disconnection
    print(f"\nSignal cleanup:")
    
    # Disconnect all receivers
    # API drift (blinker 1.9.0): `disconnect()` now REQUIRES the receiver — there is no
    # disconnect-all overload. Only logger1 is still connected (strong refs), so name it.
    lifecycle_signal.disconnect(logger1.handle_lifecycle)
    connection_signal.disconnect(logger1.handle_connection)
    
    print(f"After complete cleanup:")
    print(f"  Lifecycle signal: {len(lifecycle_signal.receivers)} receivers")
    print(f"  Connection signal: {len(connection_signal.receivers)} receivers")
    
    # Test sending to cleaned up signals
    lifecycle_signal.send('final-test', event_type='no_receivers')
    print("  ✓ Signals sent to empty receivers (no handlers called)")
    
    print("✓ Weak references and memory management completed")
    print("blinker example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blinker example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: blinker functionality test 5

In [ ]:
import importlib.metadata

import blinker
from blinker import signal, Signal, Namespace

# Example 5: Blinker package capabilities and event-driven architecture
print("Example 5: Blinker package capabilities and event-driven architecture")

try:
    print("Blinker - Fast, simple object-to-object signal dispatching system")
    
    # blinker 1.9 removed `blinker.__version__`; the installed distribution's metadata is the source of truth.
    blinker_version = importlib.metadata.version("blinker")
    assert blinker_version and blinker_version[0].isdigit(), f"unexpected blinker version {blinker_version!r}"
    print(f"Blinker version: {blinker_version}")
    
    # Key Blinker capabilities
    print("\nKey Blinker capabilities:")
    capabilities = [
        "🎯 Fast object-to-object signal dispatching",
        "🔗 Decoupled communication between components", 
        "📡 Named signals with automatic registration",
        "🎭 Multiple receivers per signal with ordering",
        "💾 Weak reference support for memory management",
        "🎨 Decorator-based signal connection",
        "📦 Signal namespaces for organization",
        "⚡ Thread-safe signal delivery"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # Core Blinker concepts
    print("\nCore Blinker concepts:")
    concepts = [
        ("Signals", "Named communication channels between objects"),
        ("Receivers", "Functions or methods that handle signal events"),
        ("Senders", "Objects that emit signals with data"),
        ("Namespaces", "Containers for organizing related signals"),
        ("Weak References", "Automatic cleanup of disconnected receivers"),
        ("Context Data", "Additional parameters passed with signals")
    ]
    
    for concept, description in concepts:
        print(f"  • {concept}: {description}")
    
    # Common use cases
    print("\nCommon Blinker use cases:")
    use_cases = [
        "🌐 Web application lifecycle events (Flask, Django)",
        "🎮 Game development event systems", 
        "📊 Model-View separation in GUI applications",
        "🔄 Plugin and extension systems",
        "📝 Logging and audit trail systems",
        "🛠️ Configuration change notifications",
        "📡 Inter-service communication",
        "🎨 UI component interaction patterns"
    ]
    
    for use_case in use_cases:
        print(f"  {use_case}")
    
    # Framework integrations
    print("\nFramework integrations:")
    integrations = [
        "• Flask: Built-in signal support for request lifecycle",
        "• Django: Custom signal system inspired by Blinker", 
        "• SQLAlchemy: Database event notifications",
        "• Celery: Task execution and result handling",
        "• Pyramid: Event system for web applications",
        "• Custom frameworks: Plugin architecture"
    ]
    
    for integration in integrations:
        print(f"  {integration}")
    
    # Event-driven architecture patterns
    print("\nEvent-driven architecture patterns:")
    patterns = [
        "Observer pattern implementation",
        "Publish-Subscribe messaging", 
        "Event sourcing and CQRS",
        "Microservice event communication",
        "Plugin system decoupling",
        "User interface component interaction"
    ]
    
    for pattern in patterns:
        print(f"  • {pattern}")
    
    # Demonstration of key features
    print(f"\nKey features demonstration:")
    
    # 1. Simple signal creation and usage
    demo_signal = signal('demo-event')
    
    @demo_signal.connect
    def demo_handler(sender, message=None, **kwargs):
        return f"Received: {message} from {sender}"
    
    demo_signal.send('demo-app', message='Hello Blinker!')
    print("  ✓ Signal creation, connection, and sending")
    
    # 2. Namespace organization
    demo_ns = Namespace()
    api_signal = demo_ns.signal('api-call')
    
    @api_signal.connect
    def api_handler(sender, endpoint=None, **kwargs):
        print(f"    API call to {endpoint}")
    
    api_signal.send('client', endpoint='/users')
    print("  ✓ Namespace organization and management")
    
    # 3. Multiple receivers
    @demo_signal.connect
    def additional_handler(sender, **kwargs):
        print(f"    Additional processing for signal from {sender}")
    
    demo_signal.send('multi-demo', message='Multiple handlers')
    print("  ✓ Multiple receivers per signal")
    
    # 4. Signal introspection
    print(f"  ✓ Signal introspection: {len(demo_signal.receivers)} receivers")
    
    # Best practices
    print(f"\nBlinker best practices:")
    best_practices = [
        "🔧 Use descriptive signal names for clarity",
        "📦 Group related signals in namespaces", 
        "💾 Prefer weak references for memory efficiency",
        "🎯 Send meaningful context data with signals",
        "🛡️ Handle exceptions in signal receivers",
        "📊 Use signals for logging and monitoring",
        "⚡ Keep signal handlers lightweight",
        "🔄 Document signal contracts and data"
    ]
    
    for practice in best_practices:
        print(f"  {practice}")
    
    # Performance characteristics
    print(f"\nPerformance characteristics:")
    performance_notes = [
        "Fast signal dispatching with minimal overhead",
        "Efficient weak reference management",
        "Thread-safe signal delivery and connection", 
        "Scales well with number of signals and receivers",
        "Low memory footprint for signal registration"
    ]
    
    for note in performance_notes:
        print(f"  • {note}")
    
    # Example application architecture
    print(f"\nExample event-driven application structure:")
    architecture_example = """
    # Application signals
    app_signals = Namespace()
    user_created = app_signals.signal('user-created')
    order_placed = app_signals.signal('order-placed')
    
    # Event handlers
    @user_created.connect
    def send_welcome_email(sender, user_id=None, **kwargs):
        # Send welcome email logic
        pass
    
    @order_placed.connect  
    def update_inventory(sender, order_data=None, **kwargs):
        # Update inventory logic
        pass
    
    @order_placed.connect
    def process_payment(sender, order_data=None, **kwargs):
        # Payment processing logic
        pass
    """
    
    print(architecture_example)
    
    print(f"\nBlinker enables clean, decoupled event-driven architectures")
    print("Perfect for building maintainable and extensible applications")
    print("blinker example 5 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"blinker example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")